In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
merged_df = pd.read_csv('data/merged_df.csv')

In [ ]:
merged_df['ND_GAIN_class'] = pd.cut(merged_df['ND_GAIN'],
                                    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                                    labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                                    include_lowest=True)

In [ ]:
from itables import show
show(merged_df)

In [ ]:
merged_df_2010 = merged_df[merged_df['Year'] == 2010]


In [ ]:
# Map ISO3 codes to match world map dataset
# Create a copy to avoid modifying the original dataframe
merged_df_2010_mapped = merged_df_2010.copy()

# Define mappings: ND-GAIN ISO3 -> World Map ISO3
iso3_mappings = {
    'SCG': ['MNE', 'SRB'],  # Serbia and Montenegro -> Montenegro and Serbia
    'ISR': ['PSE'],  # Israel -> Palestine (additional mapping)
    'SDN': ['SSD'],  # Sudan -> South Sudan (additional mapping)
    'CHN': ['TWN']   # China -> Taiwan (additional mapping)
}

# For each mapping, duplicate rows with the new ISO3 codes
additional_rows = []
for source_iso3, target_iso3_list in iso3_mappings.items():
    source_data = merged_df_2010_mapped[merged_df_2010_mapped['ISO3'] == source_iso3]
    if not source_data.empty:
        for target_iso3 in target_iso3_list:
            new_rows = source_data.copy()
            new_rows['ISO3'] = target_iso3
            additional_rows.append(new_rows)

# Concatenate all additional rows to the dataframe
if additional_rows:
    merged_df_2010_mapped = pd.concat([merged_df_2010_mapped] + additional_rows, ignore_index=True)

print(f"Original rows: {len(merged_df_2010)}")
print(f"Rows after mapping: {len(merged_df_2010_mapped)}")

In [ ]:
show(merged_df_2010_mapped)

In [ ]:
# Load world geometry data directly from Natural Earth
url = 'https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip'
world = gpd.read_file(url)

# Use ISO_A3_EH instead of ISO_A3 (ISO_A3_EH has better coverage)
# Merge the world geometry with the MAPPED data based on ISO3 codes
world_merged = world.merge(merged_df_2010_mapped, left_on='ISO_A3_EH', right_on='ISO3', how='left')

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
world_merged.plot(column='ND_GAIN_class', 
                  ax=ax, 
                  legend=True,
                  cmap='RdYlGn',
                  edgecolor='black',
                  linewidth=0.5,
                  missing_kwds={'color': 'lightgrey'})
ax.set_title('ND-GAIN Class by Country (2010)', fontsize=16)
ax.axis('off')
plt.show()

In [ ]:
# Cross-check ISO3 codes between datasets
iso3_in_ndgain = set(merged_df_2010_mapped['ISO3'].dropna().unique())
iso3_in_world = set(world['ISO_A3_EH'].dropna().unique())

# Find codes in ND-GAIN but not in world map
missing_in_world = sorted(iso3_in_ndgain - iso3_in_world)

# Find codes in world map but not in ND-GAIN
missing_in_ndgain = sorted(iso3_in_world - iso3_in_ndgain)

print(f"Total ISO3 codes in ND-GAIN dataset: {len(iso3_in_ndgain)}")
print(f"Total ISO3 codes in World map dataset: {len(iso3_in_world)}")
print(f"\n{'='*60}")
print(f"ISO3 codes in ND-GAIN but NOT in World map ({len(missing_in_world)}):")
print(missing_in_world)
print(f"\n{'='*60}")
print(f"ISO3 codes in World map but NOT in ND-GAIN ({len(missing_in_ndgain)}):")
print(missing_in_ndgain)